<a href="https://colab.research.google.com/github/talitharhmd/Tugas-Akhir/blob/main/Embedding_Cluster.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **00 Import Library**
Menginstall library yang dibutuhkan untuk proses embedding, reduksi dimensi, dan clustering.

In [1]:
# ================= INSTALL PACKAGE =================
!pip install gensim
!pip install umap-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 37.5 MB/s eta 0:00:00


In [2]:
# import libraries
import numpy as np
import pandas as pd
import ast
import os
import glob
import os
import gc

# embedding
from gensim.models import Word2Vec, FastText
from sklearn.metrics.pairwise import cosine_similarity

# reduksi dimensi
import umap
from sklearn.decomposition import PCA

# clustering
from sklearn.cluster import KMeans

# evaluasi clustering
from sklearn.metrics import silhouette_score, davies_bouldin_score

# preprocessing untuk cosine
from sklearn.preprocessing import normalize

In [3]:
import ast

# **04 Pembuatan Word Embedding**

## 4.1 Load Data Hasil LDA
Menggunakan dataset terbaik dari hasil LDA sebagai dasar embedding.

In [4]:
df = pd.read_csv('/content/lemmatization_nb5_na0.6 (3).csv')
df.head()


,doc_id,clean_text,lemmatization,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topik_dominan,prob_topik
0,0,today s digital world demands about automated ...,"['today', 'digital', 'world', 'demand', 'autom...",0.382534,0.516984,0.001629,0.002900,0.034902,0.001863,0.059188,2,0.516984
1,1,sentiment analysis sa is focused on mining opi...,"['sentiment', 'sa', 'focused', 'mining', 'opin...",0.266511,0.463015,0.000791,0.001397,0.083231,0.138796,0.046260,2,0.463015
2,2,aspect based sentiment analysis absa has recen...,"['aspect', 'sentiment', 'absa', 'recently', 'a...",0.651315,0.344010,0.000732,0.001291,0.001028,0.000838,0.000786,1,0.651315
3,3,in the era of web online forums blogs and twit...,"['era', 'web', 'online', 'forum', 'blog', 'twi...",0.001337,0.639947,0.001113,0.001969,0.270079,0.084360,0.001195,2,0.639947
4,4,financial and economic news is continuously mo...,"['financial', 'economic', 'news', 'continuousl...",0.361857,0.332745,0.000809,0.057704,0.001132,0.000926,0.244827,1,0.361857


In [5]:
# handle parsing token dengan aman
def safe_literal_eval(x):
    # handle NaN / None
    if pd.isna(x):
        return []

    # jika sudah list
    if isinstance(x, list):
        return x

    # jika string list
    try:
        return ast.literal_eval(x)
    except:
        # fallback: split biasa
        return str(x).split()

# apply ke kolom utama
df['tokens'] = df['lemmatization'].apply(safe_literal_eval)

# pastikan tidak ada token kosong
df = df[df['tokens'].map(len) > 0].reset_index(drop=True)

# set variabel utama
sentences = df['tokens']
original_text = df['clean_text']

In [6]:
sentences

,tokens
0,"[today, digital, world, demand, automated, sen..."
1,"[sentiment, sa, focused, mining, opinion, iden..."
2,"[aspect, sentiment, absa, recently, attracted,..."
3,"[era, web, online, forum, blog, twitter, becom..."
4,"[financial, economic, news, continuously, moni..."
...,...
5045,"[purpose, examines, role, analytics, managemen..."
5046,"[controlled, environment, agriculture, cea, co..."
5047,"[debate, examines, climate, change, present, c..."
5048,"[hybrid, automated, machine, automl, architect..."


In [7]:
original_text

,clean_text
0,today s digital world demands about automated ...
1,sentiment analysis sa is focused on mining opi...
2,aspect based sentiment analysis absa has recen...
3,in the era of web online forums blogs and twit...
4,financial and economic news is continuously mo...
...,...
5045,purpose this study examines the role of data a...
5046,the controlled environment agriculture cea sys...
5047,this debate examines how climate change presen...
5048,this research propose s a hybrid automated mac...


## 4.1 Menentukan Dataset untuk Embedding

In [8]:
# folder penyimpanan embedding
os.makedirs("embedding_results/word2vec", exist_ok=True)
os.makedirs("embedding_results/fasttext", exist_ok=True)
os.makedirs("embedding_results/fast2vec", exist_ok=True)

In [9]:
# folder penyimpanan embedding
os.makedirs("word_embedding_results/word2vec", exist_ok=True)
os.makedirs("word_embedding_results/fasttext", exist_ok=True)
os.makedirs("word_embedding_results/fast2vec", exist_ok=True)

## 4.2 Konfigurasi Parameter Embedding
Menentukan parameter utama untuk training Word2Vec, FastText, dan Fast2Vec.

In [10]:
# parameter embedding
vector_dimensions = [100]
context_windows = [5, 10]

model_architecture = {
    "cbow": 0,
    "skipgram": 1
}

epochs = 15
min_word_frequency = 2
seed_value = 42
num_workers = 2

# khusus fast2vec
alpha_weights = np.round(np.arange(0.1, 1.0, 0.1), 1)

# rekap eksperimen
experiment_results = []
trained_word2vec_models = {}
trained_fasttext_models = {}

## 4.3 Dokumen Embedding
Mnegubah token menjadi vektor dokumen (rata-rata kata)

In [11]:
def build_document_vector(tokens, word_vectors):
    """
    Mengubah list token menjadi vektor dokumen dengan merata-rata vektor kata.
    model_wv: KeyedVectors (model.wv) dari Gensim
    """
    word_vecs = [word_vectors[word] for word in tokens if word in word_vectors]

    if len(word_vecs) == 0:
        return np.zeros(word_vectors.vector_size)

    return np.mean(word_vecs, axis=0).astype(np.float32)

## 4.4 Word2Vec Embedding
Melatih Word2Vec menggunakan dataset terbaik dan menyimpan hasil embedding serta parameter model.

In [12]:
print("--- MULAI EKSPERIMEN WORD2VEC ---")

preview_shown = False

total_models = (
    len(vector_dimensions) *
    len(context_windows) *
    len(model_architecture)
)

print(f"\n📊 total model word2vec yang akan dibentuk: {total_models}")

# loop parameter vector size
for vector_size in vector_dimensions:
    # loop parameter window
    for window_size in context_windows:
        # loop arsitektur model (cbow / skipgram)
        for arch_name, sg_flag in model_architecture.items():

            # 1. nama eksperimen
            experiment_id = f"W2V_{arch_name}_S{vector_size}_W{window_size}"
            print(f"\n⚙️ menjalankan: {experiment_id}")

            # 2. training model word2vec
            model = Word2Vec(
                sentences=sentences,
                vector_size=vector_size,
                window=window_size,
                min_count=min_word_frequency,
                sg=sg_flag,
                epochs=epochs,
                workers=num_workers,
                seed=seed_value
            )

            trained_word2vec_models[experiment_id] = model

            # 3. document embedding
            document_vectors = np.array([
                build_document_vector(doc, model.wv)
                for doc in sentences
            ], dtype=np.float32)

            # simpan skor rekap
            experiment_results.append({
                "model": "word2vec",
                "architecture": arch_name,
                "vector_size": vector_size,
                "window": window_size,
                "alpha": None,
                "beta": None
            })

            # hasil document embedding
            df_doc_embed = pd.DataFrame(
                document_vectors,
                columns=[f"dim_{i}" for i in range(vector_size)]
            )

            # metadata dokumen
            df_doc_embed.insert(0, "doc_id", df['doc_id'])
            df_doc_embed.insert(1, "clean_text", original_text.values)
            df_doc_embed.insert(2, "text_processed", sentences.apply(lambda x: " ".join(x)))

            # metadata eksperimen
            df_doc_embed.insert(3, "architecture", arch_name)
            df_doc_embed.insert(4, "vector_size", vector_size)
            df_doc_embed.insert(5, "window", window_size)

            # preview eksperimen
            if not preview_shown:
                print(f"\n📄 preview document embedding ({experiment_id})")
                display(df_doc_embed.head(3))
                preview_shown = True

            # simpan file document embedding
            df_doc_embed.to_csv(
                f"embedding_results/word2vec/doc_embedding_{experiment_id}.csv",
                index=False
            )

            del document_vectors, df_doc_embed
            gc.collect()

--- MULAI EKSPERIMEN WORD2VEC ---

📊 total model word2vec yang akan dibentuk: 4

⚙️ menjalankan: W2V_cbow_S100_W5

📄 preview document embedding (W2V_cbow_S100_W5)


,doc_id,clean_text,text_processed,architecture,vector_size,window,dim_0,dim_1,dim_2,dim_3,...,dim_90,dim_91,dim_92,dim_93,dim_94,dim_95,dim_96,dim_97,dim_98,dim_99
0,0,today s digital world demands about automated ...,today digital world demand automated sentiment...,cbow,100,5,0.275636,-0.336439,0.281806,0.510947,...,-0.110353,0.422030,0.092907,-0.041191,0.086387,0.104363,-0.208253,0.175828,-0.569608,0.474026
1,1,sentiment analysis sa is focused on mining opi...,sentiment sa focused mining opinion identifica...,cbow,100,5,-0.169865,-0.346782,0.068902,0.717927,...,-0.118891,0.228883,-0.013980,-0.095867,0.061363,0.151990,-0.478211,0.304970,-0.524348,0.279755
2,2,aspect based sentiment analysis absa has recen...,aspect sentiment absa recently attracted incre...,cbow,100,5,-0.107693,-0.720080,0.110628,0.709941,...,-0.390768,0.149861,0.139571,-0.261960,0.081571,0.360531,-0.708905,0.307770,-0.764275,0.418865



⚙️ menjalankan: W2V_skipgram_S100_W5

⚙️ menjalankan: W2V_cbow_S100_W10

⚙️ menjalankan: W2V_skipgram_S100_W10


Menampilkan kontribusi tiap kata terhadap vektor dokumen untuk validasi manual perhitungan embedding.

In [13]:
print("--- word embedding untuk Word2Vec ---")

preview_shown = False

for experiment_id, model in trained_word2vec_models.items():

    print(f"\n⚙️ menjalankan word embedding: {experiment_id}")

    word_vectors = model.wv
    vocab = list(word_vectors.index_to_key)
    sentences_words = sentences

    # mapping word → vector
    word_matrix = {w: word_vectors[w] for w in vocab}

    # ambil metadata dari experiment_id (BENAR)
    arch = experiment_id.split("_")[1]
    vector_size = int(experiment_id.split("_S")[1].split("_")[0])
    window = experiment_id.split("_W")[-1]

    # trace per document
    def build_trace(doc_tokens):
        trace_cols = []

        for i in range(word_vectors.vector_size):

            parts = [
                f"{w}*{word_matrix[w][i]:.6f}"
                for w in doc_tokens
                if w in word_matrix
            ]

            trace_cols.append(", ".join(parts))

        return trace_cols

    trace_data = [build_trace(doc) for doc in sentences_words]

    df_trace = pd.DataFrame(trace_data)
    df_trace.columns = [f"word_dim{i}" for i in range(word_vectors.vector_size)]

    # document metadata
    df_doc = pd.DataFrame({
        "doc_id": df["doc_id"],
        "clean_text": original_text.values,
        "text_processed": sentences.apply(lambda x: " ".join(x)),
        "architecture": arch,
        "vector_size": vector_size,
        "window": window
    })

    df_final = pd.concat([df_doc, df_trace], axis=1)

    if not preview_shown:
        print(f"\n📄 preview word embedding trace ({experiment_id})")
        display(df_final.head(2))
        preview_shown = True

    df_final.to_csv(
        f"word_embedding_results/word2vec/word_embedding_trace_{experiment_id}.csv",
        index=False
    )

    del df_final, df_trace
    gc.collect()

--- word embedding untuk Word2Vec ---

⚙️ menjalankan word embedding: W2V_cbow_S100_W5

📄 preview word embedding trace (W2V_cbow_S100_W5)


,doc_id,clean_text,text_processed,architecture,vector_size,window,word_dim0,word_dim1,word_dim2,word_dim3,...,word_dim90,word_dim91,word_dim92,word_dim93,word_dim94,word_dim95,word_dim96,word_dim97,word_dim98,word_dim99
0,0,today s digital world demands about automated ...,today digital world demand automated sentiment...,cbow,100,5,"today*0.458382, digital*1.345522, world*3.5797...","today*-0.563319, digital*-0.731106, world*-0.6...","today*-0.311302, digital*0.311396, world*1.843...","today*-0.265652, digital*-0.626490, world*0.14...",...,"today*0.243226, digital*-0.230295, world*3.083...","today*0.245486, digital*0.842084, world*2.2204...","today*0.309391, digital*-0.117070, world*-1.95...","today*-0.832816, digital*-0.401884, world*-0.8...","today*-0.003359, digital*2.301523, world*1.294...","today*0.515344, digital*-1.480514, world*-1.21...","today*0.770448, digital*1.818056, world*-0.583...","today*-0.092895, digital*0.219219, world*-2.83...","today*-0.623283, digital*0.232746, world*0.204...","today*-0.200658, digital*-0.474690, world*-1.1..."
1,1,sentiment analysis sa is focused on mining opi...,sentiment sa focused mining opinion identifica...,cbow,100,5,"sentiment*-0.520728, sa*-0.386532, focused*0.4...","sentiment*-3.258090, sa*-0.625017, focused*0.3...","sentiment*0.478133, sa*-0.170839, focused*0.28...","sentiment*2.947648, sa*1.408500, focused*0.436...",...,"sentiment*-1.858012, sa*0.274308, focused*0.15...","sentiment*0.269496, sa*1.167911, focused*0.994...","sentiment*-0.244459, sa*-0.506931, focused*-0....","sentiment*-0.728475, sa*0.177106, focused*-0.0...","sentiment*-1.997706, sa*0.399556, focused*-0.3...","sentiment*0.853283, sa*0.336171, focused*0.041...","sentiment*-3.978590, sa*-1.516536, focused*-0....","sentiment*0.041570, sa*0.051674, focused*-0.26...","sentiment*-0.024959, sa*-0.231744, focused*-1....","sentiment*-0.591185, sa*-0.912045, focused*1.4..."



⚙️ menjalankan word embedding: W2V_skipgram_S100_W5

⚙️ menjalankan word embedding: W2V_cbow_S100_W10

⚙️ menjalankan word embedding: W2V_skipgram_S100_W10


## 4.5 FastText Embedding
Melatih FastText dengan parameter yang sama untuk dibandingkan dengan Word2Vec.

In [14]:
print("--- MULAI EKSPERIMEN FASTTEXT ---")

preview_shown = False

total_models = (
    len(vector_dimensions) *
    len(context_windows) *
    len(model_architecture)
)

print(f"\n📊 total model fasttext yang akan dibentuk: {total_models}")

# loop parameter vector size
for vector_size in vector_dimensions:
    # loop parameter window
    for window_size in context_windows:
        # loop arsitektur model (cbow / skipgram)
        for arch_name, sg_flag in model_architecture.items():

            # 1. nama eksperimen
            experiment_id = f"FT_{arch_name}_S{vector_size}_W{window_size}"
            print(f"\n⚙️ menjalankan: {experiment_id}")

            # 2. training fasttext model
            model = FastText(
                sentences=sentences,
                vector_size=vector_size,
                window=window_size,
                min_count=min_word_frequency,
                sg=sg_flag,
                epochs=epochs,
                workers=num_workers,
                seed=seed_value,

                # parameter subword
                min_n=3,
                max_n=6
            )

            trained_fasttext_models[experiment_id] = model

            # 3. document embedding
            document_vectors = np.array([
                build_document_vector(doc, model.wv)
                for doc in sentences
            ], dtype=np.float32)

            # 4. simpan rekap eksperimen
            experiment_results.append({
                "model": "fasttext",
                "architecture": arch_name,
                "vector_size": vector_size,
                "window": window_size,
                "alpha": None,
                "beta": None
            })

            # 5. dataframe document embedding
            df_doc_embed = pd.DataFrame(
                document_vectors,
                columns=[f"dim_{i}" for i in range(vector_size)]
            )

            # metadata dokumen
            df_doc_embed.insert(0, "doc_id", df['doc_id'])
            df_doc_embed.insert(1, "clean_text", original_text.values)
            df_doc_embed.insert(2, "text_processed", sentences.apply(lambda x: " ".join(x)))

            # metadata eksperimen
            df_doc_embed.insert(3, "architecture", arch_name)
            df_doc_embed.insert(4, "vector_size", vector_size)
            df_doc_embed.insert(5, "window", window_size)

            # 6. preview hasil pertama
            if not preview_shown:
                print(f"\n📄 preview document embedding ({experiment_id})")
                display(df_doc_embed.head(3))
                preview_shown = True

            # 7. simpan hasil
            df_doc_embed.to_csv(
                f"embedding_results/fasttext/doc_embedding_{experiment_id}.csv",
                index=False
            )

            # 8. memory cleanup
            del document_vectors, df_doc_embed
            gc.collect()

--- MULAI EKSPERIMEN FASTTEXT ---

📊 total model fasttext yang akan dibentuk: 4

⚙️ menjalankan: FT_cbow_S100_W5

📄 preview document embedding (FT_cbow_S100_W5)


,doc_id,clean_text,text_processed,architecture,vector_size,window,dim_0,dim_1,dim_2,dim_3,...,dim_90,dim_91,dim_92,dim_93,dim_94,dim_95,dim_96,dim_97,dim_98,dim_99
0,0,today s digital world demands about automated ...,today digital world demand automated sentiment...,cbow,100,5,0.521053,-0.091690,0.592672,-0.742798,...,0.536281,-1.206319,0.161135,-0.069613,-0.037816,-0.034172,0.371773,-0.515796,0.427341,-0.036084
1,1,sentiment analysis sa is focused on mining opi...,sentiment sa focused mining opinion identifica...,cbow,100,5,0.293021,-0.173066,0.173627,-0.288984,...,0.437439,-0.550900,-0.233010,0.247906,-0.525452,-0.331389,-0.019026,-0.575681,-0.130013,-0.149239
2,2,aspect based sentiment analysis absa has recen...,aspect sentiment absa recently attracted incre...,cbow,100,5,0.440959,0.032199,0.039709,-0.649122,...,0.499442,-0.924151,-0.616623,0.244300,-0.433321,0.028894,-0.204476,-0.312681,-0.156317,-0.622476



⚙️ menjalankan: FT_skipgram_S100_W5

⚙️ menjalankan: FT_cbow_S100_W10

⚙️ menjalankan: FT_skipgram_S100_W10


Menampilkan representasi embedding FastText berbasis karakter n-gram untuk mendukung interpretasi perhitungan embedding.

In [15]:
print("--- word embedding untuk FastText ---")

preview_shown = False

# generate subword manual sesuai konsep fasttext
def generate_subwords(word, min_n=3, max_n=6):
    word = f"<{word}>"
    subwords = []

    for n in range(min_n, max_n + 1):
        for i in range(len(word) - n + 1):
            subwords.append(word[i:i+n])

    return subwords

# loop semua model fasttext hasil training
for experiment_id, model in trained_fasttext_models.items():

    # ambil word vectors
    word_vectors = model.wv
    vocab = list(word_vectors.index_to_key)

    # ambil data dokumen
    sentences_words = sentences

    # parsing metadata eksperimen
    arch = experiment_id.split("_")[1]
    window = experiment_id.split("_W")[-1]

    print(f"\n⚙️ menjalankan word embedding fasttext: {experiment_id}")

    # mapping word → vector
    word_matrix = {w: word_vectors[w] for w in vocab}

    # build trace per dokumen
    def build_trace(doc_tokens):

        trace_cols = []

        for i in range(word_vectors.vector_size):

            parts = []

            for w in doc_tokens:

                if w in word_matrix:

                    # ambil subword (manual)
                    subwords = generate_subwords(w, min_n=3, max_n=6)

                    # batasi supaya tidak terlalu panjang (opsional)
                    subwords = subwords[:5]

                    parts.append(
                        f"{w}({','.join(subwords)})*{word_matrix[w][i]:.6f}"
                    )

                else:
                    parts.append(f"{w}(oov)*0.0")

            trace_cols.append(", ".join(parts))

        return trace_cols

    # generate trace seluruh dokumen
    trace_data = [build_trace(doc) for doc in sentences_words]

    df_trace = pd.DataFrame(trace_data)
    df_trace.columns = [f"word_dim{i}" for i in range(word_vectors.vector_size)]

    # metadata dokumen
    df_doc = pd.DataFrame({
        "doc_id": df["doc_id"],
        "clean_text": original_text.values,
        "text_processed": sentences.apply(lambda x: " ".join(x)),
        "architecture": arch,
        "vector_size": word_vectors.vector_size,
        "window": window
    })

    # gabungkan metadata + trace
    df_final = pd.concat([df_doc, df_trace], axis=1)

    # preview sekali saja
    if not preview_shown:
        print(f"\n📄 preview fasttext word + subword trace ({experiment_id})")
        display(df_final.head(2))
        preview_shown = True

    # simpan hasil
    df_final.to_csv(
        f"word_embedding_results/fasttext/word_embedding_trace_{experiment_id}.csv",
        index=False
    )

    # cleanup memory
    del df_final, df_trace
    gc.collect()

--- word embedding untuk FastText ---

⚙️ menjalankan word embedding fasttext: FT_cbow_S100_W5

📄 preview fasttext word + subword trace (FT_cbow_S100_W5)


,doc_id,clean_text,text_processed,architecture,vector_size,window,word_dim0,word_dim1,word_dim2,word_dim3,...,word_dim90,word_dim91,word_dim92,word_dim93,word_dim94,word_dim95,word_dim96,word_dim97,word_dim98,word_dim99
0,0,today s digital world demands about automated ...,today digital world demand automated sentiment...,cbow,100,5,"today(<to,tod,oda,day,ay>)*-0.660936, digital(...","today(<to,tod,oda,day,ay>)*-0.462538, digital(...","today(<to,tod,oda,day,ay>)*1.561529, digital(<...","today(<to,tod,oda,day,ay>)*0.844327, digital(<...",...,"today(<to,tod,oda,day,ay>)*1.257909, digital(<...","today(<to,tod,oda,day,ay>)*-1.214953, digital(...","today(<to,tod,oda,day,ay>)*0.490591, digital(<...","today(<to,tod,oda,day,ay>)*1.461841, digital(<...","today(<to,tod,oda,day,ay>)*-0.330407, digital(...","today(<to,tod,oda,day,ay>)*0.483026, digital(<...","today(<to,tod,oda,day,ay>)*-0.753672, digital(...","today(<to,tod,oda,day,ay>)*0.941733, digital(<...","today(<to,tod,oda,day,ay>)*1.264964, digital(<...","today(<to,tod,oda,day,ay>)*0.224133, digital(<..."
1,1,sentiment analysis sa is focused on mining opi...,sentiment sa focused mining opinion identifica...,cbow,100,5,"sentiment(<se,sen,ent,nti,tim)*1.690143, sa(<s...","sentiment(<se,sen,ent,nti,tim)*-0.097756, sa(<...","sentiment(<se,sen,ent,nti,tim)*2.544813, sa(<s...","sentiment(<se,sen,ent,nti,tim)*0.635019, sa(<s...",...,"sentiment(<se,sen,ent,nti,tim)*0.821998, sa(<s...","sentiment(<se,sen,ent,nti,tim)*-2.340404, sa(<...","sentiment(<se,sen,ent,nti,tim)*0.080565, sa(<s...","sentiment(<se,sen,ent,nti,tim)*-0.567647, sa(<...","sentiment(<se,sen,ent,nti,tim)*0.374873, sa(<s...","sentiment(<se,sen,ent,nti,tim)*0.501785, sa(<s...","sentiment(<se,sen,ent,nti,tim)*0.074578, sa(<s...","sentiment(<se,sen,ent,nti,tim)*-1.991806, sa(<...","sentiment(<se,sen,ent,nti,tim)*1.176329, sa(<s...","sentiment(<se,sen,ent,nti,tim)*-3.376612, sa(<..."



⚙️ menjalankan word embedding fasttext: FT_skipgram_S100_W5

⚙️ menjalankan word embedding fasttext: FT_cbow_S100_W10

⚙️ menjalankan word embedding fasttext: FT_skipgram_S100_W10


## 4.6 Fast2Vec Embedding
Menggabungkan Word2Vec dan FastText menggunakan bobot alpha dan beta.

In [16]:
print("--- MULAI EKSPERIMEN FAST2VEC ---")

preview_shown = False

# hitung total pasangan model (w2v yang punya pasangan fasttext)
paired_models = [
    w2v_key for w2v_key in trained_word2vec_models
    if w2v_key.replace("W2V_", "FT_") in trained_fasttext_models
]

total_models = len(paired_models) * len(alpha_weights)

print(f"\n📊 total model fast2vec yang akan dibentuk: {total_models}")


# fungsi fast2vec (gabungan w2v + fasttext)
def get_fast2vec_vector(tokens, w2v_wv, ft_wv, alpha):
    """
    menggabungkan vektor word2vec dan fasttext
    alpha * w2v + (1 - alpha) * fasttext
    """
    vectors = []

    for word in tokens:
        if word in w2v_wv and word in ft_wv:
            v_w2v = w2v_wv[word]
            v_ft = ft_wv[word]

            # kombinasi vektor
            v_hybrid = (alpha * v_w2v) + ((1 - alpha) * v_ft)
            vectors.append(v_hybrid)

        elif word in ft_wv:
            vectors.append(ft_wv[word])

        elif word in w2v_wv:
            vectors.append(w2v_wv[word])

    if len(vectors) == 0:
        return np.zeros(w2v_wv.vector_size)

    return np.mean(vectors, axis=0).astype(np.float32)


# loop semua model word2vec
for w2v_key, w2v_model in trained_word2vec_models.items():

    # cari pasangan fasttext dengan parameter sama
    ft_key = w2v_key.replace("W2V_", "FT_")

    if ft_key not in trained_fasttext_models:
        continue

    ft_model = trained_fasttext_models[ft_key]

    # parsing metadata dari experiment id
    arch = w2v_key.split("_")[1]
    vector_size = int(w2v_key.split("_S")[1].split("_")[0])
    window = int(w2v_key.split("_W")[-1])

    print(f"\n🔗 pasangan model: {w2v_key} + {ft_key}")

    # loop alpha
    for alpha in alpha_weights:

        beta = round(1 - alpha, 1)

        # nama eksperimen fast2vec
        f2v_id = f"F2V_{arch}_S{vector_size}_W{window}_A{alpha}_B{beta}"
        print(f"\n⚙️ menjalankan: {f2v_id}")

        # step 1: document embedding fast2vec
        document_vectors = np.array([
            get_fast2vec_vector(doc, w2v_model.wv, ft_model.wv, alpha)
            for doc in sentences
        ], dtype=np.float32)

        # step 2: simpan rekap eksperimen
        experiment_results.append({
            "model": "fast2vec",
            "architecture": arch,
            "vector_size": vector_size,
            "window": window,
            "alpha": alpha,
            "beta": beta
        })

        # step 3: dataframe embedding
        df_embedding = pd.DataFrame(
            document_vectors,
            columns=[f"dim_{i}" for i in range(vector_size)]
        )

        # metadata dokumen
        df_embedding.insert(0, "doc_id", df["doc_id"])
        df_embedding.insert(1, "clean_text", original_text.values)
        df_embedding.insert(2, "text_processed", sentences.apply(lambda x: " ".join(x)))

        # metadata eksperimen
        df_embedding.insert(3, "architecture", arch)
        df_embedding.insert(4, "vector_size", vector_size)
        df_embedding.insert(5, "window", window)
        df_embedding.insert(6, "alpha", alpha)
        df_embedding.insert(7, "beta", beta)

        # preview sekali saja
        if not preview_shown:
            print(f"\n📄 preview fast2vec embedding ({f2v_id})")
            display(df_embedding.head(3))
            preview_shown = True

        df_embedding.to_csv(
            f"embedding_results/fast2vec/doc_embedding_{f2v_id}.csv",
            index=False
        )

        del document_vectors, df_embedding
        gc.collect()

--- MULAI EKSPERIMEN FAST2VEC ---

📊 total model fast2vec yang akan dibentuk: 36

🔗 pasangan model: W2V_cbow_S100_W5 + FT_cbow_S100_W5

⚙️ menjalankan: F2V_cbow_S100_W5_A0.1_B0.9

📄 preview fast2vec embedding (F2V_cbow_S100_W5_A0.1_B0.9)


,doc_id,clean_text,text_processed,architecture,vector_size,window,alpha,beta,dim_0,dim_1,...,dim_90,dim_91,dim_92,dim_93,dim_94,dim_95,dim_96,dim_97,dim_98,dim_99
0,0,today s digital world demands about automated ...,today digital world demand automated sentiment...,cbow,100,5,0.1,0.9,0.496512,-0.116165,...,0.471617,-1.043484,0.154312,-0.066771,-0.025396,-0.020319,0.313771,-0.446633,0.327646,0.014927
1,1,sentiment analysis sa is focused on mining opi...,sentiment sa focused mining opinion identifica...,cbow,100,5,0.1,0.9,0.246882,-0.190132,...,0.381861,-0.473055,-0.211092,0.213676,-0.466740,-0.283224,-0.064636,-0.487911,-0.169128,-0.106539
2,2,aspect based sentiment analysis absa has recen...,aspect sentiment absa recently attracted incre...,cbow,100,5,0.1,0.9,0.386094,-0.043029,...,0.410422,-0.816750,-0.541003,0.193674,-0.381832,0.062057,-0.254919,-0.250636,-0.217113,-0.518342



⚙️ menjalankan: F2V_cbow_S100_W5_A0.2_B0.8

⚙️ menjalankan: F2V_cbow_S100_W5_A0.3_B0.7

⚙️ menjalankan: F2V_cbow_S100_W5_A0.4_B0.6

⚙️ menjalankan: F2V_cbow_S100_W5_A0.5_B0.5

⚙️ menjalankan: F2V_cbow_S100_W5_A0.6_B0.4

⚙️ menjalankan: F2V_cbow_S100_W5_A0.7_B0.3

⚙️ menjalankan: F2V_cbow_S100_W5_A0.8_B0.2

⚙️ menjalankan: F2V_cbow_S100_W5_A0.9_B0.1

🔗 pasangan model: W2V_skipgram_S100_W5 + FT_skipgram_S100_W5

⚙️ menjalankan: F2V_skipgram_S100_W5_A0.1_B0.9

⚙️ menjalankan: F2V_skipgram_S100_W5_A0.2_B0.8

⚙️ menjalankan: F2V_skipgram_S100_W5_A0.3_B0.7

⚙️ menjalankan: F2V_skipgram_S100_W5_A0.4_B0.6

⚙️ menjalankan: F2V_skipgram_S100_W5_A0.5_B0.5

⚙️ menjalankan: F2V_skipgram_S100_W5_A0.6_B0.4

⚙️ menjalankan: F2V_skipgram_S100_W5_A0.7_B0.3

⚙️ menjalankan: F2V_skipgram_S100_W5_A0.8_B0.2

⚙️ menjalankan: F2V_skipgram_S100_W5_A0.9_B0.1

🔗 pasangan model: W2V_cbow_S100_W10 + FT_cbow_S100_W10

⚙️ menjalankan: F2V_cbow_S100_W10_A0.1_B0.9

⚙️ menjalankan: F2V_cbow_S100_W10_A0.2_B0.8

⚙️ menj

In [17]:
print("--- word embedding untuk fast2vec (hybrid trace) ---")

preview_shown = False

# generate subword (sama seperti fasttext)
def generate_subwords(word, min_n=3, max_n=6):
    word = f"<{word}>"
    subwords = []

    for n in range(min_n, max_n + 1):
        for i in range(len(word) - n + 1):
            subwords.append(word[i:i+n])

    return subwords

# loop pasangan model
for w2v_key, w2v_model in trained_word2vec_models.items():

    ft_key = w2v_key.replace("W2V_", "FT_")

    if ft_key not in trained_fasttext_models:
        continue

    ft_model = trained_fasttext_models[ft_key]

    # parsing metadata
    arch = "cbow" if "_cbow_" in w2v_key else "skipgram"
    window = int(w2v_key.split("_W")[-1])
    vector_size = w2v_model.vector_size

    w2v_wv = w2v_model.wv
    ft_wv = ft_model.wv

    sentences_words = sentences

    # loop alpha
    for alpha in alpha_weights:

        beta = round(1 - alpha, 1)

        f2v_id = f"F2V_A{alpha}_B{beta}_{w2v_key.replace('W2V_', '')}"

        print(f"\n⚙️ menjalankan word embedding fast2vec: {f2v_id}")

        # build trace per dokumen
        def build_trace(doc_tokens):

            trace_cols = []

            for i in range(vector_size):

                parts = []

                for w in doc_tokens:

                    v_w2v = w2v_wv[w] if w in w2v_wv else None
                    v_ft = ft_wv[w] if w in ft_wv else None

                    if v_w2v is not None and v_ft is not None:

                        subwords = generate_subwords(w)[:5]

                        parts.append(
                            f"{w}[{alpha}*{v_w2v[i]:.6f} + {beta}*{v_ft[i]:.6f}]"
                        )

                    elif v_ft is not None:
                        subwords = generate_subwords(w)[:5]

                        parts.append(
                            f"{w}({','.join(subwords)})*{v_ft[i]:.6f}"
                        )

                    elif v_w2v is not None:
                        parts.append(
                            f"{w}*{v_w2v[i]:.6f}"
                        )

                    else:
                        parts.append(f"{w}(oov)*0.0")

                trace_cols.append(", ".join(parts))

            return trace_cols

        # generate trace
        trace_data = [build_trace(doc) for doc in sentences_words]

        df_trace = pd.DataFrame(trace_data)
        df_trace.columns = [f"word_dim{i}" for i in range(vector_size)]

        # metadata dokumen
        df_doc = pd.DataFrame({
            "doc_id": df["doc_id"],
            "clean_text": original_text.values,
            "text_processed": sentences.apply(lambda x: " ".join(x)),
            "architecture": arch,
            "vector_size": vector_size,
            "window": window,
            "alpha": alpha,
            "beta": beta
        })

        # gabungkan
        df_final = pd.concat([df_doc, df_trace], axis=1)

        # preview sekali
        if not preview_shown:
            print(f"\n📄 preview fast2vec trace ({f2v_id})")
            display(df_final.head(2))
            preview_shown = True

        # simpan
        df_final.to_csv(
            f"word_embedding_results/fast2vec/word_embedding_trace_{f2v_id}.csv",
            index=False
        )

        # cleanup
        del df_final, df_trace
        gc.collect()

--- word embedding untuk fast2vec (hybrid trace) ---

⚙️ menjalankan word embedding fast2vec: F2V_A0.1_B0.9_cbow_S100_W5

📄 preview fast2vec trace (F2V_A0.1_B0.9_cbow_S100_W5)


,doc_id,clean_text,text_processed,architecture,vector_size,window,alpha,beta,word_dim0,word_dim1,...,word_dim90,word_dim91,word_dim92,word_dim93,word_dim94,word_dim95,word_dim96,word_dim97,word_dim98,word_dim99
0,0,today s digital world demands about automated ...,today digital world demand automated sentiment...,cbow,100,5,0.1,0.9,"today[0.1*0.458382 + 0.9*-0.660936], digital[0...","today[0.1*-0.563319 + 0.9*-0.462538], digital[...",...,"today[0.1*0.243226 + 0.9*1.257909], digital[0....","today[0.1*0.245486 + 0.9*-1.214953], digital[0...","today[0.1*0.309391 + 0.9*0.490591], digital[0....","today[0.1*-0.832816 + 0.9*1.461841], digital[0...","today[0.1*-0.003359 + 0.9*-0.330407], digital[...","today[0.1*0.515344 + 0.9*0.483026], digital[0....","today[0.1*0.770448 + 0.9*-0.753672], digital[0...","today[0.1*-0.092895 + 0.9*0.941733], digital[0...","today[0.1*-0.623283 + 0.9*1.264964], digital[0...","today[0.1*-0.200658 + 0.9*0.224133], digital[0..."
1,1,sentiment analysis sa is focused on mining opi...,sentiment sa focused mining opinion identifica...,cbow,100,5,0.1,0.9,"sentiment[0.1*-0.520728 + 0.9*1.690143], sa[0....","sentiment[0.1*-3.258090 + 0.9*-0.097756], sa[0...",...,"sentiment[0.1*-1.858012 + 0.9*0.821998], sa[0....","sentiment[0.1*0.269496 + 0.9*-2.340404], sa[0....","sentiment[0.1*-0.244459 + 0.9*0.080565], sa[0....","sentiment[0.1*-0.728475 + 0.9*-0.567647], sa[0...","sentiment[0.1*-1.997706 + 0.9*0.374873], sa[0....","sentiment[0.1*0.853283 + 0.9*0.501785], sa[0.1...","sentiment[0.1*-3.978590 + 0.9*0.074578], sa[0....","sentiment[0.1*0.041570 + 0.9*-1.991806], sa[0....","sentiment[0.1*-0.024959 + 0.9*1.176329], sa[0....","sentiment[0.1*-0.591185 + 0.9*-3.376612], sa[0..."



⚙️ menjalankan word embedding fast2vec: F2V_A0.2_B0.8_cbow_S100_W5

⚙️ menjalankan word embedding fast2vec: F2V_A0.3_B0.7_cbow_S100_W5

⚙️ menjalankan word embedding fast2vec: F2V_A0.4_B0.6_cbow_S100_W5


KeyboardInterrupt: 

## 4.7 Rekap Hasil Embedding
Menampilkan seluruh konfigurasi model embedding yang telah dijalankan.

In [18]:
df_rekap = pd.DataFrame(experiment_results)
df_rekap.columns = df_rekap.columns.str.lower()

expected_cols = [
    "model",
    "dataset",
    "architecture",
    "vector_size",
    "window",
    "alpha",
    "beta"
]

for col in expected_cols:
    if col not in df_rekap.columns:
        df_rekap[col] = np.nan
df_rekap = df_rekap[expected_cols]

# hitung jumlah model
total_all = len(df_rekap)
total_per_model = df_rekap["model"].value_counts().reset_index()
total_per_model.columns = ["model", "jumlah_model"]

# detail kombinasi
df_unique = df_rekap.drop_duplicates()


print("\n📊 Ringkasan Model Embedding")
display(df_rekap.head(10))

print(f"\n Total seluruh model: {total_all}")

print("\n Jumlah model per metode:")
display(total_per_model)

print("\n Kombinasi unik eksperimen:")
display(df_unique.head(10))


📊 Ringkasan Model Embedding


,model,dataset,architecture,vector_size,window,alpha,beta
0,word2vec,NaN,cbow,100,5,NaN,NaN
1,word2vec,NaN,skipgram,100,5,NaN,NaN
2,word2vec,NaN,cbow,100,10,NaN,NaN
3,word2vec,NaN,skipgram,100,10,NaN,NaN
4,fasttext,NaN,cbow,100,5,NaN,NaN
5,fasttext,NaN,skipgram,100,5,NaN,NaN
6,fasttext,NaN,cbow,100,10,NaN,NaN
7,fasttext,NaN,skipgram,100,10,NaN,NaN
8,fast2vec,NaN,cbow,100,5,0.1,0.9
9,fast2vec,NaN,cbow,100,5,0.2,0.8



 Total seluruh model: 44

 Jumlah model per metode:


,model,jumlah_model
0,fast2vec,36
1,word2vec,4
2,fasttext,4



 Kombinasi unik eksperimen:


,model,dataset,architecture,vector_size,window,alpha,beta
0,word2vec,NaN,cbow,100,5,NaN,NaN
1,word2vec,NaN,skipgram,100,5,NaN,NaN
2,word2vec,NaN,cbow,100,10,NaN,NaN
3,word2vec,NaN,skipgram,100,10,NaN,NaN
4,fasttext,NaN,cbow,100,5,NaN,NaN
5,fasttext,NaN,skipgram,100,5,NaN,NaN
6,fasttext,NaN,cbow,100,10,NaN,NaN
7,fasttext,NaN,skipgram,100,10,NaN,NaN
8,fast2vec,NaN,cbow,100,5,0.1,0.9
9,fast2vec,NaN,cbow,100,5,0.2,0.8


# **05 Integrasi Fitur**

In [19]:
# folder untuk menyimpan hasil integrasi fitur
os.makedirs("embedding_results/combined_features", exist_ok=True)

## 5.1 Ambil Fitur LDA


In [20]:
df_lda = df.copy()

# ambil kolom topik (otomatis, fleksibel kalau jumlah topik berubah)
topic_cols = [col for col in df_lda.columns if col.startswith("topic_")]

# ambil matrix lda
lda_vectors = df_lda[topic_cols].values

print("jumlah topik:", len(topic_cols))
print("shape lda vector:", lda_vectors.shape)

# simpan juga metadata penting
df_lda_meta = df_lda[["doc_id", "clean_text"]].copy()

display(df_lda.head(2))

jumlah topik: 7
shape lda vector: (5050, 7)


,doc_id,clean_text,lemmatization,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topik_dominan,prob_topik,tokens
0,0,today s digital world demands about automated ...,"['today', 'digital', 'world', 'demand', 'autom...",0.382534,0.516984,0.001629,0.002900,0.034902,0.001863,0.059188,2,0.516984,"[today, digital, world, demand, automated, sen..."
1,1,sentiment analysis sa is focused on mining opi...,"['sentiment', 'sa', 'focused', 'mining', 'opin...",0.266511,0.463015,0.000791,0.001397,0.083231,0.138796,0.046260,2,0.463015,"[sentiment, sa, focused, mining, opinion, iden..."


## 5.3 Integrasi Fitur (LDA + Embedding)
Menggabungkan vektor LDA dan embedding menjadi satu fitur gabungan.

In [21]:
# ambil semua file embedding
embedding_files = glob.glob("embedding_results/word2vec/*.csv") + \
                  glob.glob("embedding_results/fasttext/*.csv") + \
                  glob.glob("embedding_results/fast2vec/*.csv")

print(f"jumlah file embedding: {len(embedding_files)}")

jumlah file embedding: 44


In [22]:
combined_count = 0

for file in embedding_files:

    print(f"\nprocessing: {file}")

    df_embed = pd.read_csv(file)

    # pastikan urutan sesuai doc_id
    df_embed = df_embed.sort_values("doc_id").reset_index(drop=True)
    df_lda_sorted = df_lda.sort_values("doc_id").reset_index(drop=True)

    # ambil kolom embedding (dim_*)
    vector_cols = [c for c in df_embed.columns if c.startswith("dim_")]

    if len(vector_cols) == 0:
        print("skip: tidak ada kolom embedding dim_*")
        continue

    embedding_vectors = df_embed[vector_cols].values

    # validasi jumlah dokumen
    if len(lda_vectors) != len(embedding_vectors):
        print("warning: jumlah dokumen tidak sama, skip")
        continue

    # gabungkan fitur
    combined_vectors = np.concatenate(
        (lda_vectors, embedding_vectors),
        axis=1
    )

    print(f"shape gabungan: {combined_vectors.shape}")

    # buat dataframe
    df_combined = pd.DataFrame(combined_vectors)

    # rename kolom → topic + embedding
    topic_names = [f"topic_{i}" for i in range(len(topic_cols))]
    embed_names = [f"emb_{i}" for i in range(len(vector_cols))]

    df_combined.columns = topic_names + embed_names

    # metadata
    df_combined.insert(0, "doc_id", df_embed["doc_id"])
    df_combined.insert(1, "clean_text", df_embed["clean_text"])
    df_combined.insert(2, "text_processed", df_embed["text_processed"])
    df_combined.insert(3, "lda_topics", len(topic_cols))

    # simpan
    base_name = os.path.basename(file).replace(".csv", "")
    output_path = f"embedding_results/combined_features/COMBINED_{base_name}.csv"

    df_combined.to_csv(output_path, index=False)

    print(f"saved → {output_path}")
    combined_count += 1

print(f"\njumlah file hasil integrasi: {combined_count}")


processing: embedding_results/word2vec/doc_embedding_W2V_cbow_S100_W10.csv
shape gabungan: (5050, 107)
saved → embedding_results/combined_features/COMBINED_doc_embedding_W2V_cbow_S100_W10.csv

processing: embedding_results/word2vec/doc_embedding_W2V_skipgram_S100_W5.csv
shape gabungan: (5050, 107)
saved → embedding_results/combined_features/COMBINED_doc_embedding_W2V_skipgram_S100_W5.csv

processing: embedding_results/word2vec/doc_embedding_W2V_skipgram_S100_W10.csv
shape gabungan: (5050, 107)
saved → embedding_results/combined_features/COMBINED_doc_embedding_W2V_skipgram_S100_W10.csv

processing: embedding_results/word2vec/doc_embedding_W2V_cbow_S100_W5.csv
shape gabungan: (5050, 107)
saved → embedding_results/combined_features/COMBINED_doc_embedding_W2V_cbow_S100_W5.csv

processing: embedding_results/fasttext/doc_embedding_FT_skipgram_S100_W10.csv
shape gabungan: (5050, 107)
saved → embedding_results/combined_features/COMBINED_doc_embedding_FT_skipgram_S100_W10.csv

processing: embe

## 5.4 Preview Hasil Integrasi Fitur

- Pada tahap ini ditampilkan salah satu hasil penggabungan antara vektor distribusi topik LDA dan vektor embedding dokumen.

- Tujuannya untuk memastikan bahwa proses concatenation sudah berjalan
dengan benar dan struktur datanya sesuai.

Preview ini hanya menampilkan beberapa baris pertama dari salah satu
file hasil eksperimen.

In [23]:
combined_files = glob.glob("embedding_results/combined_features/*.csv")

# pilih file pertama sebagai contoh
sample_file = combined_files[0]

print(f"\nMenampilkan contoh file:")
print(sample_file)

df_sample = pd.read_csv(sample_file)
display(df_sample.head())


Menampilkan contoh file:
embedding_results/combined_features/COMBINED_doc_embedding_F2V_skipgram_S100_W10_A0.3_B0.7.csv


,doc_id,clean_text,text_processed,lda_topics,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,...,emb_90,emb_91,emb_92,emb_93,emb_94,emb_95,emb_96,emb_97,emb_98,emb_99
0,0,today s digital world demands about automated ...,today digital world demand automated sentiment...,7,0.382534,0.516984,0.001629,0.002900,0.034902,0.001863,...,-0.029827,0.011037,-0.049381,0.038314,-0.021853,0.059877,0.080547,0.036939,0.046775,0.028336
1,1,sentiment analysis sa is focused on mining opi...,sentiment sa focused mining opinion identifica...,7,0.266511,0.463015,0.000791,0.001397,0.083231,0.138796,...,0.017545,0.084835,-0.103008,0.107697,-0.059073,0.024389,0.019616,-0.001266,-0.039837,0.007003
2,2,aspect based sentiment analysis absa has recen...,aspect sentiment absa recently attracted incre...,7,0.651315,0.344010,0.000732,0.001291,0.001028,0.000838,...,-0.021798,0.106006,-0.190609,0.055033,-0.063769,0.016825,0.053219,0.038078,-0.093471,0.017133
3,3,in the era of web online forums blogs and twit...,era web online forum blog twitter becoming pri...,7,0.001337,0.639947,0.001113,0.001969,0.270079,0.084360,...,-0.031002,0.058516,-0.075851,0.096975,-0.094408,0.126933,0.042837,0.034675,-0.069342,0.023441
4,4,financial and economic news is continuously mo...,financial economic news continuously monitored...,7,0.361857,0.332745,0.000809,0.057704,0.001132,0.000926,...,0.026299,0.079557,-0.054126,0.046114,-0.043032,0.077978,0.043004,-0.025336,-0.046763,-0.002614


# **05 Reduksi Dimensi**
Mengambil semua file hasil integrasi gabungan LDA + embedding yang akan dipakai untuk eksperimen.

In [24]:
combined_files = [
    f"embedding_results/combined_features/{f}"
    for f in os.listdir("embedding_results/combined_features")
    if f.endswith(".csv")
]

dicoba 3

In [25]:
print(f"jumlah file combined: {len(combined_files)}")

# parameter umap
n_components_list = [5, 10]   # sesuai kebutuhan clustering karena tidak terlalu besar
n_neighbors_list = [10, 15]   # literatur: lokal vs balance
min_dist_list = [0.1, 0.3]    # rapat vs agak menyebar

umap_results = {}

jumlah file combined: 44


## 5.1 UMAP
Mengubah data ke dimensi lebih kecil pakai UMAP dengan cosine distance.

In [26]:
print("--- mulai proses UMAP ---")

preview_count = 0
max_preview = 3

for file in combined_files:

    print(f"\n📂 processing: {file}")

    df_combined = pd.read_csv(file)

    # pisahkan metadata
    meta_cols = ["doc_id", "clean_text", "text_processed", "lda_topics"]
    df_meta = df_combined[meta_cols]

    # ambil fitur numerik
    feature_cols = [c for c in df_combined.columns if c not in meta_cols]
    X = df_combined[feature_cols].values

    # normalisasi (cosine)
    X_norm = normalize(X)

    for n_comp in n_components_list:
        for n_neigh in n_neighbors_list:
            for min_d in min_dist_list:

                exp_id = f"{os.path.basename(file).replace('.csv','')}_UMAP_C{n_comp}_N{n_neigh}_D{min_d}"

                reducer = umap.UMAP(
                    n_components=n_comp,
                    n_neighbors=n_neigh,
                    min_dist=min_d,
                    metric="cosine",
                    random_state=42
                )

                X_umap = reducer.fit_transform(X_norm)

                print(f"✔ {exp_id} → shape: {X_umap.shape}")

                # ===== buat dataframe hasil =====
                umap_cols = [f"umap_{i+1}" for i in range(n_comp)]
                df_umap = pd.DataFrame(X_umap, columns=umap_cols)

                # gabungkan dengan metadata
                df_final = pd.concat([df_meta.reset_index(drop=True),
                                      df_umap.reset_index(drop=True)], axis=1)

                # simpan ke dict
                umap_results[exp_id] = df_final

                # (opsional) simpan ke file
                # df_final.to_csv(f"umap_results/{exp_id}.csv", index=False)

                # preview terbatas
                if preview_count < max_preview:
                    print(f"\n📄 preview hasil UMAP: {exp_id}")
                    display(df_final.head(3))
                    preview_count += 1

--- mulai proses UMAP ---

📂 processing: embedding_results/combined_features/COMBINED_doc_embedding_F2V_skipgram_S100_W10_A0.3_B0.7.csv


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


✔ COMBINED_doc_embedding_F2V_skipgram_S100_W10_A0.3_B0.7_UMAP_C5_N10_D0.1 → shape: (5050, 5)

📄 preview hasil UMAP: COMBINED_doc_embedding_F2V_skipgram_S100_W10_A0.3_B0.7_UMAP_C5_N10_D0.1


,doc_id,clean_text,text_processed,lda_topics,umap_1,umap_2,umap_3,umap_4,umap_5
0,0,today s digital world demands about automated ...,today digital world demand automated sentiment...,7,10.619962,8.907569,5.479846,3.103794,4.466532
1,1,sentiment analysis sa is focused on mining opi...,sentiment sa focused mining opinion identifica...,7,10.745676,9.081290,5.577899,2.884119,4.086594
2,2,aspect based sentiment analysis absa has recen...,aspect sentiment absa recently attracted incre...,7,10.573461,9.753963,4.764270,3.587504,4.156340


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


...

✔ COMBINED_doc_embedding_W2V_cbow_S100_W10_UMAP_C10_N10_D0.3 → shape: (5050, 10)


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


✔ COMBINED_doc_embedding_W2V_cbow_S100_W10_UMAP_C10_N15_D0.1 → shape: (5050, 10)


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


✔ COMBINED_doc_embedding_W2V_cbow_S100_W10_UMAP_C10_N15_D0.3 → shape: (5050, 10)


In [27]:
# folder output
os.makedirs("embedding_results/umap", exist_ok=True)

saved_count = 0

for exp_id, X_umap in umap_results.items():

    df_umap = pd.DataFrame(X_umap, columns=[f"umap_{i}" for i in range(X_umap.shape[1])])

    df_umap.to_csv(f"embedding_results/umap/{exp_id}.csv", index=False)
    saved_count += 1

print(f"\njumlah file umap tersimpan: {saved_count}")


jumlah file umap tersimpan: 352


### PCA
Mengurangi dimensi pakai PCA

In [ ]:
# from sklearn.decomposition import PCA
# from sklearn.preprocessing import StandardScaler

# pca_results = {}

# n_components_list = [5, 10, 30, 50]

# for file in combined_files:

#     print(f"\nPCA Processing: {file}")

#     df = pd.read_csv(file)

#     # ambil fitur numerik saja
#     feature_cols = [c for c in df.columns if c not in
#                     ["doc_id", "clean_text", "text_processed", "dataset", "lda_topics"]]

#     X = df[feature_cols].values

#     # scaling dulu → penting untuk PCA
#     scaler = StandardScaler()
#     X_scaled = scaler.fit_transform(X)

#     for n in n_components_list:

#         # buat PCA
#         pca = PCA(n_components=n, random_state=42)

#         # reduksi dimensi
#         X_pca = pca.fit_transform(X_scaled)

#         # simpan hasil
#         key = f"{file}_PCA_{n}"
#         pca_results[key] = X_pca

#         print(f"PCA n_components={n} → shape: {X_pca.shape}")

# **06 Clustering**

# **06 Cluster K-Means**
Digunakan untuk mencoba beberapa jumlah cluster (k) dan menghitung nilai Silhouette & DBI dengan pendekatan 'cosine'

In [ ]:
import numpy as np
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

def kmeans_cosine(X, k, max_iter=100, tol=1e-4, random_state=42):

    np.random.seed(random_state)

    X_norm = normalize(X)
    n_samples = X_norm.shape[0]

    # inisialisasi centroid
    random_idx = np.random.choice(n_samples, k, replace=False)
    centroids = X_norm[random_idx]

    for _ in range(max_iter):

        # similarity
        sim = cosine_similarity(X_norm, centroids)

        # assign
        labels = np.argmax(sim, axis=1)

        new_centroids = []

        for i in range(k):
            cluster_points = X_norm[labels == i]

            if len(cluster_points) == 0:
                new_centroids.append(X_norm[np.random.randint(0, n_samples)])
            else:
                new_centroids.append(np.mean(cluster_points, axis=0))

        new_centroids = normalize(np.array(new_centroids))

        # konvergensi
        if np.linalg.norm(centroids - new_centroids) < tol:
            break

        centroids = new_centroids

    return labels, centroids

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score

def evaluate_kmeans_umap(df_umap, range_k=(2,10)):

    # ambil hanya kolom umap
    umap_cols = [c for c in df_umap.columns if "umap_" in c]
    X = df_umap[umap_cols].values

    results = []

    for k in range(range_k[0], range_k[1]+1):

        labels, centroids = kmeans_cosine(X, k)

        sil = silhouette_score(normalize(X), labels, metric='cosine')
        dbi = davies_bouldin_score(normalize(X), labels)

        print(f"k={k} | Silhouette={sil:.4f} | DBI={dbi:.4f}")

        results.append({
            "k": k,
            "silhouette": sil,
            "dbi": dbi
        })

    return results

In [ ]:
all_cluster_results = {}

for key, df_umap in umap_results.items():

    print(f"\n📂 Clustering: {key}")

    results = evaluate_kmeans_umap(df_umap)

    # pilih k terbaik (silhouette tertinggi)
    best_k = max(results, key=lambda x: x["silhouette"])["k"]

    print(f"✅ Best k: {best_k}")

    # ambil fitur
    umap_cols = [c for c in df_umap.columns if "umap_" in c]
    X = df_umap[umap_cols].values

    labels, _ = kmeans_cosine(X, best_k)

    # gabungkan ke dataframe
    df_result = df_umap.copy()
    df_result["cluster"] = labels

    all_cluster_results[key] = df_result

    display(df_result.head(3))


📂 Clustering: COMBINED_doc_embedding_W2V_cbow_S100_W10_UMAP_C5_N10_D0.1
k=2 | Silhouette=0.7862 | DBI=0.6697
k=3 | Silhouette=0.6854 | DBI=0.9863
k=4 | Silhouette=0.6483 | DBI=0.8514
k=5 | Silhouette=0.6180 | DBI=0.8587
k=6 | Silhouette=0.5898 | DBI=0.9185
k=7 | Silhouette=0.5331 | DBI=1.0297
k=8 | Silhouette=0.6197 | DBI=0.8566
k=9 | Silhouette=0.6003 | DBI=0.8809
k=10 | Silhouette=0.5985 | DBI=0.9147
✅ Best k: 2


,doc_id,clean_text,text_processed,lda_topics,umap_1,umap_2,umap_3,umap_4,umap_5,cluster
0,0,today s digital world demands about automated ...,today digital world demand automated sentiment...,7,2.869109,8.445724,4.164492,4.281650,6.208026,1
1,1,sentiment analysis sa is focused on mining opi...,sentiment sa focused mining opinion identifica...,7,2.083688,8.859046,4.842039,4.557263,6.228567,1
2,2,aspect based sentiment analysis absa has recen...,aspect sentiment absa recently attracted incre...,7,1.632093,8.845299,4.971497,4.641635,6.023027,1



📂 Clustering: COMBINED_doc_embedding_W2V_cbow_S100_W10_UMAP_C5_N10_D0.3
k=2 | Silhouette=0.6966 | DBI=0.8631
k=3 | Silhouette=0.5552 | DBI=1.0615
k=4 | Silhouette=0.4804 | DBI=1.2063
k=5 | Silhouette=0.4639 | DBI=1.1475
k=6 | Silhouette=0.4761 | DBI=1.1556
k=7 | Silhouette=0.5389 | DBI=1.0625
k=8 | Silhouette=0.5552 | DBI=0.9776
k=9 | Silhouette=0.5443 | DBI=0.9756
k=10 | Silhouette=0.5369 | DBI=1.0455
✅ Best k: 2


,doc_id,clean_text,text_processed,lda_topics,umap_1,umap_2,umap_3,umap_4,umap_5,cluster
0,0,today s digital world demands about automated ...,today digital world demand automated sentiment...,7,3.481082,8.338076,3.690452,4.319672,7.555686,1
1,1,sentiment analysis sa is focused on mining opi...,sentiment sa focused mining opinion identifica...,7,2.787600,8.647925,4.724242,4.408253,7.829095,1
2,2,aspect based sentiment analysis absa has recen...,aspect sentiment absa recently attracted incre...,7,2.416262,8.447720,5.011697,4.095253,8.095634,1



📂 Clustering: COMBINED_doc_embedding_W2V_cbow_S100_W10_UMAP_C5_N15_D0.1
k=2 | Silhouette=0.8018 | DBI=0.6028
k=3 | Silhouette=0.6322 | DBI=0.9498
k=4 | Silhouette=0.5561 | DBI=1.0379
k=5 | Silhouette=0.6144 | DBI=0.8381
k=6 | Silhouette=0.6279 | DBI=0.8323
k=7 | Silhouette=0.4602 | DBI=1.0973
k=8 | Silhouette=0.6241 | DBI=0.8439
k=9 | Silhouette=0.6099 | DBI=0.9036
k=10 | Silhouette=0.5981 | DBI=0.9180
✅ Best k: 2


...


📂 Clustering: COMBINED_doc_embedding_F2V_cbow_S300_W10_A0.2_B0.8_UMAP_C5_N10_D0.1
k=2 | Silhouette=0.6885 | DBI=0.9995
k=3 | Silhouette=0.7739 | DBI=0.6627
k=4 | Silhouette=0.5952 | DBI=0.8514
k=5 | Silhouette=0.5527 | DBI=0.8947
k=6 | Silhouette=0.5732 | DBI=0.9035
k=7 | Silhouette=0.5602 | DBI=0.8672
k=8 | Silhouette=0.5609 | DBI=0.8821
k=9 | Silhouette=0.5450 | DBI=0.9179
k=10 | Silhouette=0.5462 | DBI=0.9668
✅ Best k: 3


,doc_id,clean_text,text_processed,lda_topics,umap_1,umap_2,umap_3,umap_4,umap_5,cluster
0,0,today s digital world demands about automated ...,today digital world demand automated sentiment...,7,2.876990,4.077609,1.728917,3.112761,1.602740,2
1,1,sentiment analysis sa is focused on mining opi...,sentiment sa focused mining opinion identifica...,7,2.358354,4.829746,1.963493,3.366391,1.445907,2
2,2,aspect based sentiment analysis absa has recen...,aspect sentiment absa recently attracted incre...,7,2.087731,5.001047,2.143334,3.532134,1.223971,2



📂 Clustering: COMBINED_doc_embedding_F2V_cbow_S300_W10_A0.2_B0.8_UMAP_C5_N10_D0.3
k=2 | Silhouette=0.5374 | DBI=1.3564
k=3 | Silhouette=0.5999 | DBI=0.9397
k=4 | Silhouette=0.5556 | DBI=0.9889
k=5 | Silhouette=0.5431 | DBI=0.9826
k=6 | Silhouette=0.5165 | DBI=0.9767
k=7 | Silhouette=0.5265 | DBI=0.9761
k=8 | Silhouette=0.4783 | DBI=1.0849
k=9 | Silhouette=0.5140 | DBI=1.0451
k=10 | Silhouette=0.5182 | DBI=1.0797
✅ Best k: 3


,doc_id,clean_text,text_processed,lda_topics,umap_1,umap_2,umap_3,umap_4,umap_5,cluster
0,0,today s digital world demands about automated ...,today digital world demand automated sentiment...,7,3.539884,3.485415,1.624361,3.242872,2.268714,1
1,1,sentiment analysis sa is focused on mining opi...,sentiment sa focused mining opinion identifica...,7,3.384003,4.448293,2.228446,3.711924,2.223110,1
2,2,aspect based sentiment analysis absa has recen...,aspect sentiment absa recently attracted incre...,7,3.075609,4.670344,2.646086,3.907409,2.155782,1



📂 Clustering: COMBINED_doc_embedding_F2V_cbow_S300_W10_A0.2_B0.8_UMAP_C5_N15_D0.1
k=2 | Silhouette=0.6894 | DBI=0.8032
k=3 | Silhouette=0.5646 | DBI=0.9866
k=4 | Silhouette=0.5815 | DBI=0.9922


# **Cluster HDBSCAN**